In [1]:
#Importando bibliotecas
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import GridSearchCV

In [2]:
#Função que plota as comparações entre valores reais e preditos
def plot_dtc_reg_comparison(df,log_comp='DTC_Linear_Reg'):
    
    fig,axes = plt.subplots(nrows=1, ncols=4, figsize=(9,15))

    y_lim = (np.floor((df.DEPTH_MD.min()-100)/50)*50, np.ceil((df.DEPTH_MD.max()+1)/50)*50)
    step = 50

    for ax in axes:
        ax.set_yticks(np.arange(y_lim[0], y_lim[1], step))
        ax.set_ylim(y_lim[0], y_lim[1])
        ax.invert_yaxis()
        ax.grid()
        
    axes[0].plot(df.GR,df.DEPTH_MD,lw=0.8,color='black',label='GR')
    axes[0].set_xlim(0,150)
    axes[0].set_xlabel('GR (uApi)',fontsize=16)

    axes[0].set_ylabel('Depth (m)',fontsize=16)

    #Adicionando um eixo gêmeo para o caliper
    axes0 = axes[0].twiny()
    axes0.plot(df.CALI,df.DEPTH_MD,lw=0.8,color='red',ls='--',label='CALI')
    axes0.set_xlabel('CALI (in)',fontsize=16)
    axes0.set_xlim(6,18)
    axes0.legend(loc='upper left',fontsize=8)
        

    axes[1].semilogx(df.RDEP,df.DEPTH_MD,lw=0.8,color='black',label='ILD')
    axes[1].semilogx(df.RMED,df.DEPTH_MD,color='red',lw=0.8,ls='--',label='ILM')

    axes[1].set_xlabel(r'$Res$ ($\Omega \cdot m$)',fontsize=16)
    axes[1].set_xlim(1, 1000)
    axes[1].set_xticks([1,10,100,1000])
    axes[1].grid(which='both',axis='x')


    axes[2].plot(df.DTC,df.DEPTH_MD,lw=0.8,color='black',label='DT')
    axes[2].set_xlim(140,40)
    axes[2].set_xlabel(r'$\Delta t$ ($\mu$s/ft)',fontsize=16)

    axes[2].plot(df[log_comp],df.DEPTH_MD,lw=0.8,color='RED',label='DT',ls='--')


    axes[3].plot(df.RHOB,df.DEPTH_MD,lw=0.8,color='black',label='RHOB')
    axes[3].set_xlim(2,3)
    axes[3].set_xlabel(r'g/cm$^3$',fontsize=16)

    fig.tight_layout()
    
    return fig, axes

In [3]:
dataset = pd.read_csv('Dados/train_dataset_proc.csv',index_col=0)
blind = pd.read_csv('Dados/blind_dataset_proc.csv',index_col=0)

#Separando o conjunto em teste e treino

columns = dataset.drop(['LITHOLOGY','GROUP','WELL','DEPTH_MD','DTC'], axis=1).columns

X = dataset[columns]
y = dataset['DTC']

#Separando em treino e teste matendo as proporções das classes relacionadas à litologia
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,
                                                    stratify=dataset['FORCE_2020_LITHOFACIES_LITHOLOGY'])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

##### Arquitetura adotada

<img src="mlp.png" width="800">



In [4]:
#Aplicando o KNN Regressor sem normalização

mlp = MLPRegressor(hidden_layer_sizes=(10,5,2), 
                   max_iter=500, 
                   random_state=42,
                   activation='relu')

mlp.fit(X_train, y_train)

#Aplicando o KNN Regressor com normalização
mlp_scaled = MLPRegressor(hidden_layer_sizes=(10,5,2), 
                          max_iter=500, 
                          random_state=42,
                          activation='relu')

mlp_scaled.fit(X_train_scaled, y_train)

print('MLP sem normalização')
print('---'*20)
print('Log do treino')
print(f'Erro médio absoluto (MAE): {mean_absolute_error(y_train, mlp.predict(X_train)):.2f}')
print(f'Erro quadrático médio (MSE): {mean_squared_error(y_train, mlp.predict(X_train)):.2f}')
print(f'Coeficiente de determinação (R²): {r2_score(y_train, mlp.predict(X_train)):.2f}')
print('---'*20)
print('Log do teste')
print(f'Erro médio absoluto (MAE): {mean_absolute_error(y_test, mlp.predict(X_test)):.2f}')
print(f'Erro quadrático médio (MSE): {mean_squared_error(y_test, mlp.predict(X_test)):.2f}')
print(f'Coeficiente de determinação (R²): {r2_score(y_test, mlp.predict(X_test)):.2f}')
print('---'*20)
print('\n')

print('MLP com normalização')
print('---'*20)
print('Log do treino')
print(f'Erro médio absoluto (MAE): {mean_absolute_error(y_train, mlp_scaled.predict(X_train_scaled)):.2f}')
print(f'Erro quadrático médio (MSE): {mean_squared_error(y_train, mlp_scaled.predict(X_train_scaled)):.2f}')
print(f'Coeficiente de determinação (R²): {r2_score(y_train, mlp_scaled.predict(X_train_scaled)):.2f}')
print('---'*20)
print('Log do teste')
print(f'Erro médio absoluto (MAE): {mean_absolute_error(y_test, mlp_scaled.predict(X_test_scaled)):.2f}')
print(f'Erro quadrático médio (MSE): {mean_squared_error(y_test, mlp_scaled.predict(X_test_scaled)):.2f}')
print(f'Coeficiente de determinação (R²): {r2_score(y_test, mlp_scaled.predict(X_test_scaled)):.2f}')
print('---'*20)



MLP sem normalização
------------------------------------------------------------
Log do treino
Erro médio absoluto (MAE): 10.45
Erro quadrático médio (MSE): 207.76
Coeficiente de determinação (R²): 0.74
------------------------------------------------------------
Log do teste
Erro médio absoluto (MAE): 10.46
Erro quadrático médio (MSE): 208.94
Coeficiente de determinação (R²): 0.74
------------------------------------------------------------


MLP com normalização
------------------------------------------------------------
Log do treino
Erro médio absoluto (MAE): 5.70
Erro quadrático médio (MSE): 70.68
Coeficiente de determinação (R²): 0.91
------------------------------------------------------------
Log do teste
Erro médio absoluto (MAE): 5.71
Erro quadrático médio (MSE): 73.28
Coeficiente de determinação (R²): 0.91
------------------------------------------------------------


In [ ]:
from ann_visualizer.visualize import ann_viz
ann_viz(mlp, view=True, title="MLP Sketch - Scikit-learn")


In [ ]:
!conda install -c conda-forge python-graphviz -y

